In [1]:
import json
import pandas as pd

In [ ]:
with open('tabfact.jsonl') as f:
    data = [json.loads(line) for line in f]

In [3]:
len(data)

1000

In [4]:
df = pd.DataFrame(data)
df.head()

,id,title,context,table,query,label
0,36c13137,"Burton, Ohio",,"{'header': ['Historical population', 'Historic...",Produce a one-sentence description for each hi...,"[Burton's population was 1,452 at the 2010 cen..."
1,fc15312b,GE U28C,,"{'header': ['Railroad', 'Quantity', 'Road numb...",Produce a one-sentence description for each hi...,"[A total of 28, U28Cs were built as Burlington..."
2,1957993d,"Fife Lake, Michigan",,"{'header': ['Historical population', 'Historic...",Produce a one-sentence description for each hi...,"[The population was 443 at the 2010 census., T..."
3,231d9972,Britain Simons,,"{'header': ['Year', 'Title', 'Role', 'Notes'],...",Produce a one-sentence description for each hi...,[Britain Simons appeared in another Lifetime T...
4,9325d4bf,List of winners of the National Book Award,,"{'header': ['1964', 'Aileen Ward', 'John Keats...",Produce a one-sentence description for each hi...,[Pauline Kael's Deeper into Movies won the Nat...


In [5]:
print(df["table"].apply(lambda x: json.dumps(x, sort_keys=False)).nunique())

995


In [6]:
print(df["id"].apply(lambda x: json.dumps(x, sort_keys=False)).nunique())

1000


In [7]:
dup_mask = df["table"].apply(lambda x: json.dumps(x, sort_keys=False)).duplicated(keep=False)
dup_count = dup_mask.sum()
print("Total rows that are part of duplicated tables:", dup_count)

Total rows that are part of duplicated tables: 9


In [8]:
dup_mask = df["table"].apply(lambda x: json.dumps(x, sort_keys=False)).duplicated()
dup_count = dup_mask.sum()
print("Dups (Keey=First):", dup_count)

Dups (Keey=First): 5


In [9]:
import json
import pandas as pd

# 1) Make a stable, comparable key for each table (dicts are unhashable)
df["table_key"] = df["table"].apply(lambda x: json.dumps(x, sort_keys=True))

# 2) (Optional) keep a copy of the original IDs
df["id_original"] = df["id"]

# 3) Build a mapping from each unique table_key to the FIRST seen id
first_id_map = (
    df.drop_duplicates(subset="table_key", keep="first")[["table_key", "id"]]
      .set_index("table_key")["id"]
)

# 4) Update IDs: every row gets the first occurrence's id for its table
df["id"] = df["table_key"].map(first_id_map)

# 5) (Optional) sanity checks
total = len(df)
unique_tables = df["table_key"].nunique()
updated = (df["id"] != df["id_original"]).sum()
print(f"Total samples: {total}")
print(f"Unique tables: {unique_tables}")
print(f"IDs updated (duplicates): {updated}")

Total samples: 1000
Unique tables: 995
IDs updated (duplicates): 5


In [10]:
df.drop(['table_key', 'id_original'], axis=1, inplace=True)

In [11]:
print(df["table"].apply(lambda x: json.dumps(x, sort_keys=False)).nunique())

995


In [12]:
print(df["id"].apply(lambda x: json.dumps(x, sort_keys=False)).nunique())

995


In [13]:
df.to_json("totto.json", orient="records", force_ascii=False, indent=4)